# FTP and Copay: shared config and feature builder (NBA 3.1)

The one config for the FTP / copay block (readiness plan `60_ftp_copay_nba_3_1_readiness_plan.md`, D6 and V9). Every FTP / copay notebook `%run`s it, so there is one point-in-time rule, one join and one column list.

**Notebooks in this block**

| Notebook | Plan section | Gate |
| --- | --- | --- |
| `00_ftp_copay_config` (this one) | 4, 5, V9 | - |
| `61_ftp_copay_phase0_validation` | Phase 0, V1 - V10 | G0 |
| `23_ftp_copay_features` | Phase 1, C1 - C6 | G1 |
| `62_ftp_copay_phase2_extended_candidates` | Phase 2, X1 - X8 | (E8) |
| `63_ftp_copay_experiments` | Section 7, E0 - E8 | G2 |
| `64_ftp_copay_phase3_operations_and_gates` | Phase 3, O1 - O5, S1 - S7, open questions | G3 |

**Decision log** (my decisions, taken where the plan left a choice; each is tested or reported in the notebooks)

| ID | Decision | Why |
| --- | --- | --- |
| DEC-01 | Cohorts = `promotional` family (day-17 anchor, c1 - c28) | `00_cohort_generation` lists FTP in the promotional family and the block lives in `04_Promotional_Features`. The assessment's S4 numbers were made on the `claims` windows, so S4 is reconciled on the claims windows separately (61, V2b) |
| DEC-02 | Point-in-time: effective end E = `SVC_DT` - `TRIM_DAYS`; a claim is known if `RX_FILL_DATE` <= E and `INSERT_TS` < `SVC_DT` + 1 day; window N = (E - N months, E] | Plan D2, D4, principle 3. The window is shifted, not shortened, so the 1M window is still a full month |
| DEC-03 | `TRIM_DAYS` = 14 | D4 recommendation; V3 measured 95.1% FTP / 96.9% copay load within 14 days; E7 compares 0 / 14 / 21 |
| DEC-04 | Link = universe `PROVIDER_ID` -> `NPI_NUMBER` (3.0 `kerendia_nba_3_0_npi_bhid_mapping`, the `00_cohort_config` default) -> `PRSC_NPI`; claims with no `PRSC_NPI` dropped and counted | Principle 7 / V4. `BH_ID` is only cross-checked |
| DEC-05 | Universe = `HCP_Universe_without_call_hcps` (294,727) | Current 3.1 universe used by all 3.1 blocks |
| DEC-06 | Recency sentinel = 9999 days | Above any real gap (history starts 2021), never 0 (principle 6) |
| DEC-07 | Band edges 0 / 1 / 2 / 4 / 10 / 25 -> band 0 - 5 on 12M fills | 67% of FTP HCPs have 1 fill; copay median ~4, long tail |
| DEC-08 | C5 uses `NET_PTNT_COPAY` only inside [0, 2000] USD; HCPs with no copay fill get 0 (the `CPY_HAS_ANY_12M` flag tells them apart) | Netted copay has negative and > 2,000 values (min -2,135, max 2,860) from clawback netting (D9) |
| DEC-09 | Caps (D5) = training p99 of 12M fills among active HCP x cohort rows (c1 - c24): FTP 21, copay 53 | Measured in 61 V6 on the 3.1 promotional cohorts (plan example 23 / 45 was on a different window) |
| DEC-10 | Shipped C2 skew variants: 12M capped + 12M banded per program; the log variant is computed only inside E3 | Keeps the block at 27 columns (plan: 25 - 30) |
| DEC-11 | Output keys follow the 3.1 field-activity block: `HCP_COHORT_ID, PROVIDER_ID, COHORT, DATASET, PREDICTION_MONTH, SVC_DT` + `PULL_DT`; Delta under `dev_pipeline/notebook_output/market_access_nba_3_1/` | Same join keys as the other promotional outputs |
| DEC-12 | NPIs shared by 2 universe providers keep the fill on both (counted in V4) | Same as the other 3.1 NPI joins; affects 11 NPIs (61 V4) |

In [0]:
from pyspark.sql import functions as F, Window
import json, datetime as _dt

# ---------------- Sources (D1: UC claims tables are the single feature source) ----------------
CATALOG = "ph-com-ai-us-prod-catalog-2553213839599380"
PROGRAM_TABLES = {
    "FTP": f"`{CATALOG}`.market_access.kerendia_ftp_claims",
    "CPY": f"`{CATALOG}`.market_access.kerendia_copay_claims",
}
# O5: the config owns the program group lists (must equal kerendia_data_needs/Copay_FTP/00_common)
PROGRAM_GROUPS = {
    "FTP": ["101194196WF", "112194196WF", "151194196WF", "161194196WF", "171194196WF", "181194196WF", "191194196WF"],
    "CPY": ["1BJEOFSFLDF", "121194196DF", "131194196DF"],
}
NBA31_VOLUME   = f"/Volumes/{CATALOG}/kerendia_nba_3/kerendia_nba_3_1_volume"
UNIVERSE_PATH  = f"{NBA31_VOLUME}/HCP_Universe_without_call_hcps"                       # DEC-05, 294,727 PROVIDER_IDs
MAPPING_PATH   = (f"/Volumes/{CATALOG}/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/"
                  "ops_pipeline/query_output/kerendia_nba_3_0_npi_bhid_mapping")          # DEC-04 (00_cohort_config default)
COHORT_FAMILY  = "promotional"                                                           # DEC-01
COHORT_PATH    = f"{NBA31_VOLUME}/cohorts/{COHORT_FAMILY}/cohorts.parquet"
CLAIMS_COHORT_PATH = f"{NBA31_VOLUME}/cohorts/claims/cohorts.parquet"                    # S4 reconciliation only
OUTPUT_DIR     = f"{NBA31_VOLUME}/hcp_level_model/dev_pipeline/notebook_output/market_access_nba_3_1"
FILL_EVENTS_PATH = f"{OUTPUT_DIR}/ftp_copay_fill_events"
FEATURES_PATH    = f"{OUTPUT_DIR}/kerendia_nba_3_1_hcp_model_ftp_copay_features"
EXTENDED_PATH    = f"{OUTPUT_DIR}/kerendia_nba_3_1_hcp_model_ftp_copay_extended_candidates"
QC_DIR           = f"{OUTPUT_DIR}/qc"
# Experiment inputs (63): baseline features and target
PRIOR_WRITING_PATH = f"{NBA31_VOLUME}/hcp_level_model/dev_pipeline/notebook_output/hcp_features/kerendia_prior_writing"
MONTHLY_RX_PATH    = f"{NBA31_VOLUME}/hcp_level_model/dev_pipeline/query_output/kerendia_claims_monthly_from_raw_fct_rx"

# ---------------- Run parameters (V9 / S5: the column list is generated from these) ----------------
WINDOWS_MONTHS        = [1, 3, 6, 12]        # D3
TRIM_DAYS             = 14                   # D4 / DEC-03 (E7 tests 0 / 14 / 21)
RECENCY_SENTINEL_DAYS = 9999                 # DEC-06
CAPS_12M              = {"FTP": 21, "CPY": 53}   # D5 / DEC-09 (training p99, measured in 61 V6)
BAND_EDGES            = [0, 1, 2, 4, 10, 25]     # DEC-07 -> band 0..5
COPAY_HIGH_USD        = 300                  # C5 share threshold
COPAY_VALID_RANGE     = (0.0, 2000.0)        # DEC-08
FEATURE_DATE_COL      = "RX_FILL_DATE"       # D2 (E6 negative control uses RX_WRITTEN_DATE)
DATA_START_DATE       = "2021-01-01"
REFRESH_MAX_AGE_DAYS  = 7                    # O1
SOURCE_STALE_DAYS     = 21                   # O1 (same as build QC)
MAX_ROW_DROP_PCT      = 5.0                  # O1 (same as build QC)
KEYS = ["HCP_COHORT_ID", "PROVIDER_ID", "COHORT", "DATASET", "PREDICTION_MONTH", "SVC_DT"]   # DEC-11
EXCLUDED_COLUMNS = ["IS_BENEFIT_CLAWED_BACK", "IS_ZERO_NET_BENEFIT", "MDM_SPECIALTY", "RX_WRITTEN_DATE"]  # V5: never features

def feature_columns(programs=("FTP", "CPY"), windows=None):
    """S5: ordered feature column list, generated from the config only."""
    windows = windows or WINDOWS_MONTHS
    cols = []
    for p in programs:
        cols.append(f"{p}_HAS_ANY_12M")                                   # C1
        cols += [f"{p}_FILLS_{w}M" for w in windows]                      # C2 raw
        cols += [f"{p}_FILLS_12M_CAP", f"{p}_FILLS_12M_BAND"]             # C2 skew variants (D5)
        cols += [f"{p}_NEWRX_FILLS_{w}M" for w in windows]                # C3
        cols.append(f"{p}_DAYS_SINCE_LAST_FILL")                          # C4
    if "CPY" in programs:
        cols += ["CPY_AVG_COPAY_BEFORE_CARD_12M", "CPY_COPAY_GT300_SHARE_12M"]  # C5
    if "FTP" in programs and "CPY" in programs:
        cols.append("FTP_CPY_BOTH_12M")                                    # C6
    return cols

# ---------------- Loaders ----------------
def load_cohorts(path=None):
    c = spark.read.parquet(path or COHORT_PATH)
    return (c.select(F.col("cohort_id").alias("COHORT"), F.col("dataset").alias("DATASET"),
                     F.col("prediction_month").alias("PREDICTION_MONTH"),
                     F.col("lookback_start_date").alias("LOOKBACK_START_DATE"),
                     F.col("lookback_end_date").alias("SVC_DT"))
             .withColumn("COHORT_NUM", F.expr("cast(substring(COHORT, 2) as int)")))

def load_universe_npi():
    """V4: universe PROVIDER_ID -> NPI_NUMBER (+ mapping BH_ID, used for the cross-check only)."""
    u = spark.read.parquet(UNIVERSE_PATH).select("PROVIDER_ID").distinct()
    m = (spark.read.format("delta").load(MAPPING_PATH)
         .select("PROVIDER_ID", "NPI_NUMBER", F.col("BH_ID").alias("MAP_BH_ID")))
    return u, u.join(m, "PROVIDER_ID", "left")

def raw_claims(program):
    return spark.table(PROGRAM_TABLES[program]).withColumn("PROGRAM", F.lit(program))

EVENT_COLUMNS = ["PROGRAM", "ORGN_CLM_AUTH_NUM", "PRSC_NPI", "BH_ID", "RX_FILL_DATE", "RX_WRITTEN_DATE",
                 "INSERT_TS", "IS_NEW_RX", "FILL_NUMBER", "RX_NUM", "DAY_SUPPLY_NUM", "NDC_NUM",
                 "NET_PTNT_BNFT", "NET_PTNT_COPAY", "OTHR_CVRG_CD", "PHMCY_NABP", "PAT_AGE", "PAT_GNDR", "MMBR_ID"]

def build_fill_events(programs=("FTP", "CPY")):
    """Section 5 intermediate ftp_copay_fill_events: one row per claim x program x universe HCP.
    MMBR_ID / patient columns stay in this intermediate only and never reach a feature output (Risk: PII)."""
    _, um = load_universe_npi()
    um = um.filter(F.col("NPI_NUMBER").isNotNull())
    out = None
    for p in programs:
        c = raw_claims(p).filter(F.col("PRSC_NPI").isNotNull()).select(*EVENT_COLUMNS)
        e = c.join(um, c.PRSC_NPI == um.NPI_NUMBER, "inner").drop("NPI_NUMBER")
        out = e if out is None else out.unionByName(e)
    return out.filter(F.col("RX_FILL_DATE") >= DATA_START_DATE)

def _band(colname):
    expr = F.lit(-1)
    for k, edge in enumerate(BAND_EDGES):
        expr = F.when(F.col(colname) >= edge, F.lit(k)).otherwise(expr)
    return expr

def known_events(events, cohorts, trim_days=None, date_col=None, insert_filter=True):
    """S1 point-in-time rule (DEC-02): events x cohorts restricted to what is known at each cohort."""
    trim_days = TRIM_DAYS if trim_days is None else trim_days
    date_col = date_col or FEATURE_DATE_COL
    cc = cohorts.withColumn("EFF_END", F.date_sub("SVC_DT", trim_days)).select("COHORT", "SVC_DT", "EFF_END")
    x = events.withColumn("FD", F.col(date_col)).crossJoin(F.broadcast(cc)).filter(F.col("FD") <= F.col("EFF_END"))
    if insert_filter:
        x = x.filter(F.col("INSERT_TS") < F.date_add("SVC_DT", 1).cast("timestamp"))
    return x

def build_core_features(events, cohorts, trim_days=None, date_col=None, insert_filter=True,
                        programs=("FTP", "CPY"), windows=None):
    """Phase 1 core block C1-C6 (sparse: only HCP x cohort with a known claim)."""
    windows = windows or WINDOWS_MONTHS
    x = known_events(events.select("PROVIDER_ID", "PROGRAM", "ORGN_CLM_AUTH_NUM", "RX_FILL_DATE", "RX_WRITTEN_DATE",
                                   "INSERT_TS", "IS_NEW_RX", "NET_PTNT_COPAY"),
                     cohorts, trim_days, date_col, insert_filter)
    for w in windows:
        x = x.withColumn(f"IN{w}", (F.col("FD") > F.add_months("EFF_END", -w)).cast("int"))
    lo, hi = COPAY_VALID_RANGE
    valid_cp = (F.col("IN12") == 1) & F.col("NET_PTNT_COPAY").between(lo, hi)
    aggs = [F.max("FD").alias("LAST_FD")]
    for w in windows:
        aggs.append(F.sum(f"IN{w}").alias(f"FILLS_{w}M"))
        aggs.append(F.sum(F.col(f"IN{w}") * F.col("IS_NEW_RX").cast("int")).alias(f"NEWRX_FILLS_{w}M"))
    aggs += [F.avg(F.when(valid_cp, F.col("NET_PTNT_COPAY"))).alias("AVG_COPAY_12M"),
             F.avg(F.when(valid_cp, (F.col("NET_PTNT_COPAY") > COPAY_HIGH_USD).cast("int"))).alias("GT300_SHARE_12M")]
    g = x.groupBy("PROVIDER_ID", "COHORT", "EFF_END", "PROGRAM").agg(*aggs)
    wide = None
    for p in programs:
        gp = g.filter(F.col("PROGRAM") == p)
        sel = [F.col("PROVIDER_ID"), F.col("COHORT"), (F.col("FILLS_12M") > 0).cast("int").alias(f"{p}_HAS_ANY_12M")]
        sel += [F.col(f"FILLS_{w}M").alias(f"{p}_FILLS_{w}M") for w in windows]
        sel += [F.least(F.col("FILLS_12M"), F.lit(CAPS_12M[p])).alias(f"{p}_FILLS_12M_CAP")]
        sel += [F.col(f"NEWRX_FILLS_{w}M").alias(f"{p}_NEWRX_FILLS_{w}M") for w in windows]
        sel += [F.datediff("EFF_END", "LAST_FD").alias(f"{p}_DAYS_SINCE_LAST_FILL")]
        if p == "CPY":
            sel += [F.col("AVG_COPAY_12M").alias("CPY_AVG_COPAY_BEFORE_CARD_12M"),
                    F.col("GT300_SHARE_12M").alias("CPY_COPAY_GT300_SHARE_12M")]
        gp = gp.select(*sel)
        wide = gp if wide is None else wide.join(gp, ["PROVIDER_ID", "COHORT"], "full")
    return wide

def assemble_full(wide, cohorts, programs=("FTP", "CPY"), windows=None):
    """S3: every universe HCP x cohort; 0 = no fill (from config); recency sentinel; bands."""
    windows = windows or WINDOWS_MONTHS
    u = spark.read.parquet(UNIVERSE_PATH).select("PROVIDER_ID").distinct()
    sk = u.crossJoin(F.broadcast(cohorts.select("COHORT", "DATASET", "PREDICTION_MONTH", "SVC_DT")))
    df = sk.join(wide, ["PROVIDER_ID", "COHORT"], "left")
    zero_cols = [c for c in wide.columns if c not in ("PROVIDER_ID", "COHORT") and "DAYS_SINCE" not in c]
    df = df.fillna(0, subset=zero_cols)
    for p in programs:
        df = df.fillna(RECENCY_SENTINEL_DAYS, subset=[f"{p}_DAYS_SINCE_LAST_FILL"])
        df = df.withColumn(f"{p}_FILLS_12M_BAND", _band(f"{p}_FILLS_12M"))
    if "FTP" in programs and "CPY" in programs:
        df = df.withColumn("FTP_CPY_BOTH_12M", (F.col("FTP_HAS_ANY_12M") * F.col("CPY_HAS_ANY_12M")).cast("int"))
    df = df.withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "COHORT")).withColumn("PULL_DT", F.current_date())
    return df.select(*KEYS, *feature_columns(programs, windows), "PULL_DT")

def save_qc(name, records):
    """Persist a list of dict check results to QC_DIR/<name> (Delta) so the gate notebook can read them."""
    import pandas as _pd
    pdf = _pd.DataFrame(records).astype(str)
    spark.createDataFrame(pdf).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{QC_DIR}/{name}")

print(f"FTP/copay config loaded: cohorts={COHORT_FAMILY}, windows={WINDOWS_MONTHS}, trim={TRIM_DAYS}d, "
      f"caps={CAPS_12M}, {len(feature_columns())} feature columns")